# Lab 01: Create an Evaluation Strategy Charter

## 🎯 Objectives

Define evaluation scope and governance for a real AI agent scenario. By the end of this lab, you will have a complete **Evaluation Strategy Charter** that defines:
- What to evaluate and why
- Who owns evaluation decisions
- Release criteria and thresholds
- Escalation and feedback loops

## 💡 Why GenAI Evaluation Is Different

Traditional software testing relies on deterministic outputs — given input X, expect output Y. GenAI systems are:
- **Non-deterministic**: Same input can produce different outputs
- **Multi-dimensional**: Quality, safety, relevance, groundedness, task completion
- **Subjective**: No single "correct" answer for many queries
- **Context-dependent**: Performance varies by scenario, user type, data freshness

## 📋 The 5 Customer Questions Every Evaluation Strategy Must Answer

| # | Question | Maps To |
|---|----------|--------|
| 1 | What should I evaluate? | Scope & dimensions |
| 2 | What data and evaluators do I need? | Datasets & evaluator selection |
| 3 | How do I know "good enough"? | Thresholds & release criteria |
| 4 | How do I prevent regressions? | CI/CD gates |
| 5 | How do I monitor production? | Continuous evaluation |

## ⏱️ Duration: 30-45 minutes

## Step 1: Define Your Business Scenario

For this accelerator, we use the **Contoso Enterprise Support Agent** scenario:

| Attribute | Value |
|-----------|-------|
| Agent Name | Contoso Enterprise Support Agent |
| Primary Users | Internal employees seeking HR/IT/Policy information |
| Capabilities | RAG over policy docs, ticket management tool, employee profile lookup |
| Criticality | Medium-High (affects employee productivity, handles PII) |
| Deployment | Azure AI Foundry with Application Insights |

### ✏️ Exercise: Define YOUR Scenario

Replace the values below with your customer's actual scenario:

In [ ]:
# Define your evaluation scenario
evaluation_scenario = {
    "agent_name": "Contoso Enterprise Support Agent",
    "primary_users": "Internal employees (HR, IT, Operations)",
    "capabilities": [
        "RAG over policy documents",
        "Ticket creation and lookup",
        "Employee profile/authorization verification",
        "Multi-turn conversation handling",
        "Escalation to human agents"
    ],
    "criticality": "Medium-High",
    "data_sensitivity": "Contains PII (employee names, IDs, department info)",
    "regulatory_requirements": ["Data privacy", "Information accuracy", "Audit trail"],
    "deployment_target": "Azure AI Foundry"
}

print("📋 Evaluation Scenario Defined:")
for key, value in evaluation_scenario.items():
    print(f"   {key}: {value}")

## Step 2: Identify High-Risk Failure Modes

Rank failure modes by **severity** (impact if it occurs) and **likelihood** (how often it might occur).

| Severity | Description |
|----------|-------------|
| Critical | Safety incident, compliance violation, data breach |
| High | Wrong action taken, incorrect information acted upon |
| Medium | Incomplete answer, poor user experience |
| Low | Formatting issues, verbose responses |

In [ ]:
# Define failure modes ranked by severity
failure_modes = [
    {
        "id": "FM-01",
        "failure": "Agent provides incorrect policy information",
        "severity": "High",
        "likelihood": "Medium",
        "impact": "Employee makes wrong decision based on bad info",
        "evaluator_category": "RAG Quality (Groundedness, Relevance)"
    },
    {
        "id": "FM-02",
        "failure": "Agent leaks PII from one employee to another",
        "severity": "Critical",
        "likelihood": "Low",
        "impact": "Privacy violation, potential regulatory action",
        "evaluator_category": "Safety (Sensitive data leakage)"
    },
    {
        "id": "FM-03",
        "failure": "Agent calls wrong tool or provides wrong parameters",
        "severity": "High",
        "likelihood": "Medium",
        "impact": "Wrong ticket created, wrong employee record accessed",
        "evaluator_category": "Agent Process (Tool Selection, Tool Input Accuracy)"
    },
    {
        "id": "FM-04",
        "failure": "Agent generates harmful or offensive content",
        "severity": "Critical",
        "likelihood": "Low",
        "impact": "Trust erosion, HR escalation, brand damage",
        "evaluator_category": "Safety (Violence, Hate, Sexual)"
    },
    {
        "id": "FM-05",
        "failure": "Agent fails to escalate high-risk scenarios",
        "severity": "High",
        "likelihood": "Medium",
        "impact": "Critical issues unresolved, employee safety at risk",
        "evaluator_category": "Agent Outcome (Task Adherence)"
    },
    {
        "id": "FM-06",
        "failure": "Agent provides incomplete answers",
        "severity": "Medium",
        "likelihood": "High",
        "impact": "Multiple follow-ups needed, reduced productivity",
        "evaluator_category": "RAG Quality (Response Completeness)"
    },
]

import pandas as pd
df = pd.DataFrame(failure_modes)
print("🚨 Failure Modes (ranked by severity):")
display(df.sort_values('severity'))

## Step 3: Define Quality Dimensions & Priorities

Not all quality dimensions are equally important. Classify as:
- **Must-Have (Blocking)**: Failure here blocks release
- **Should-Have (Advisory)**: Tracked and reported, doesn't block
- **Nice-to-Have (Informational)**: Measured for improvement tracking

In [ ]:
# Define quality dimensions with priority
quality_dimensions = [
    {"dimension": "Safety (no harmful content)", "priority": "Must-Have", "threshold": "Zero defects", "evaluators": "Violence, Sexual, Hate, Self-harm"},
    {"dimension": "Groundedness (answers from docs)", "priority": "Must-Have", "threshold": "≥4.0/5.0", "evaluators": "Groundedness, Groundedness Pro"},
    {"dimension": "Tool Selection Accuracy", "priority": "Must-Have", "threshold": "≥90% correct", "evaluators": "Tool Call Accuracy, Tool Selection"},
    {"dimension": "Task Adherence", "priority": "Must-Have", "threshold": "≥4.0/5.0", "evaluators": "Task Adherence"},
    {"dimension": "Response Completeness", "priority": "Should-Have", "threshold": "≥3.5/5.0", "evaluators": "Response Completeness"},
    {"dimension": "Fluency", "priority": "Should-Have", "threshold": "≥4.0/5.0", "evaluators": "Fluency"},
    {"dimension": "Coherence", "priority": "Nice-to-Have", "threshold": "≥3.5/5.0", "evaluators": "Coherence"},
    {"dimension": "Response Latency", "priority": "Should-Have", "threshold": "<5 seconds p95", "evaluators": "Custom (latency check)"},
]

df_quality = pd.DataFrame(quality_dimensions)
print("📊 Quality Dimensions:")
display(df_quality)

## Step 4: Define Release Criteria

Release criteria define the **gates** that must pass before deployment.

Two types of gates:
- **Absolute**: Score must meet a fixed threshold (e.g., Groundedness ≥ 4.0)
- **Relative**: Score must not regress from baseline (e.g., no more than 5% drop)

In [ ]:
# Define release criteria
release_criteria = {
    "blocking_gates": [
        {"evaluator": "Violence", "type": "absolute", "threshold": "0 defects", "action_on_fail": "Block release"},
        {"evaluator": "Sexual", "type": "absolute", "threshold": "0 defects", "action_on_fail": "Block release"},
        {"evaluator": "Hate/Unfairness", "type": "absolute", "threshold": "0 defects", "action_on_fail": "Block release"},
        {"evaluator": "Groundedness", "type": "absolute", "threshold": "≥4.0/5.0", "action_on_fail": "Block release + human review"},
        {"evaluator": "Tool Selection", "type": "absolute", "threshold": "≥90% accuracy", "action_on_fail": "Block release"},
        {"evaluator": "Task Adherence", "type": "relative", "threshold": "No drop >5% from baseline", "action_on_fail": "Block release"},
    ],
    "advisory_gates": [
        {"evaluator": "Fluency", "type": "absolute", "threshold": "≥4.0/5.0", "action_on_fail": "Warn + track"},
        {"evaluator": "Response Completeness", "type": "relative", "threshold": "No drop >10%", "action_on_fail": "Warn + track"},
        {"evaluator": "Coherence", "type": "absolute", "threshold": "≥3.5/5.0", "action_on_fail": "Log only"},
    ],
    "human_review_triggers": [
        "Any safety evaluator flags a defect",
        "Groundedness score < 3.0 on any individual test case",
        "More than 3 tool selection errors in a run",
        "New failure pattern not seen in previous runs"
    ]
}

print("🚦 Release Criteria:")
print(f"\n   Blocking Gates ({len(release_criteria['blocking_gates'])})")
for gate in release_criteria['blocking_gates']:
    print(f"   🔴 {gate['evaluator']}: {gate['threshold']} [{gate['type']}]")

print(f"\n   Advisory Gates ({len(release_criteria['advisory_gates'])})")
for gate in release_criteria['advisory_gates']:
    print(f"   🟡 {gate['evaluator']}: {gate['threshold']} [{gate['type']}]")

print(f"\n   Human Review Triggers ({len(release_criteria['human_review_triggers'])})")
for trigger in release_criteria['human_review_triggers']:
    print(f"   👤 {trigger}")

## Step 5: Define Ownership (RACI)

| Role | R=Responsible | A=Accountable | C=Consulted | I=Informed |
|------|--------------|---------------|-------------|------------|

Clear ownership prevents "nobody owns evaluation" failure mode.

In [ ]:
# Define RACI matrix
raci_matrix = {
    "tasks": [
        {"task": "Define evaluation criteria", "product_owner": "A", "ai_engineer": "R", "rai_security": "C", "business_sme": "C", "platform": "I"},
        {"task": "Maintain golden dataset", "product_owner": "I", "ai_engineer": "R", "rai_security": "C", "business_sme": "A", "platform": "I"},
        {"task": "Set safety thresholds", "product_owner": "C", "ai_engineer": "C", "rai_security": "A/R", "business_sme": "C", "platform": "I"},
        {"task": "Run pre-release evaluation", "product_owner": "I", "ai_engineer": "R", "rai_security": "I", "business_sme": "I", "platform": "A"},
        {"task": "Approve release (blocking gate fail)", "product_owner": "A", "ai_engineer": "R", "rai_security": "C", "business_sme": "I", "platform": "I"},
        {"task": "Monitor production scores", "product_owner": "I", "ai_engineer": "R", "rai_security": "I", "business_sme": "I", "platform": "A"},
        {"task": "Red-team testing", "product_owner": "I", "ai_engineer": "C", "rai_security": "R/A", "business_sme": "I", "platform": "I"},
        {"task": "Quarterly threshold review", "product_owner": "A", "ai_engineer": "R", "rai_security": "R", "business_sme": "C", "platform": "I"},
    ]
}

df_raci = pd.DataFrame(raci_matrix['tasks'])
df_raci.columns = ['Task', 'Product Owner', 'AI Engineer', 'RAI/Security', 'Business SME', 'Platform']
print("👥 RACI Matrix:")
display(df_raci)

## Step 6: Generate the Evaluation Charter Document

Compile all decisions into a single charter document that serves as the source of truth.

In [ ]:
import json
from datetime import datetime

# Generate the charter document
charter = {
    "title": "Evaluation Strategy Charter",
    "version": "1.0",
    "date": datetime.now().strftime("%Y-%m-%d"),
    "scenario": evaluation_scenario,
    "failure_modes": failure_modes,
    "quality_dimensions": quality_dimensions,
    "release_criteria": release_criteria,
    "raci": raci_matrix,
    "evaluation_lifecycle": {
        "phases": [
            "Design: Define what to evaluate based on failure modes",
            "Development: Build datasets and configure evaluators",
            "Pre-deployment: Run full evaluation suite, establish baseline",
            "CI/CD: Automated gates on every change",
            "Production: Continuous monitoring and scoring",
            "Continuous Improvement: Feedback → dataset refresh → threshold update"
        ]
    },
    "review_cadence": "Quarterly threshold review + monthly dataset refresh"
}

# Save charter as JSON
charter_path = "./evaluation-charter.json"
with open(charter_path, "w") as f:
    json.dump(charter, f, indent=2)

print(f"✅ Evaluation Charter saved to {charter_path}")
print(f"\n📄 Charter Summary:")
print(f"   Agent: {charter['scenario']['agent_name']}")
print(f"   Failure Modes: {len(charter['failure_modes'])}")
print(f"   Quality Dimensions: {len(charter['quality_dimensions'])}")
print(f"   Blocking Gates: {len(charter['release_criteria']['blocking_gates'])}")
print(f"   Advisory Gates: {len(charter['release_criteria']['advisory_gates'])}")
print(f"   Human Review Triggers: {len(charter['release_criteria']['human_review_triggers'])}")

## Step 7: Generate Markdown Charter for Sharing

Export the charter as a human-readable markdown document for stakeholder review.

In [ ]:
# Generate markdown version
charter_md = f"""# Evaluation Strategy Charter

**Version:** {charter['version']}  
**Date:** {charter['date']}  
**Agent:** {charter['scenario']['agent_name']}  
**Criticality:** {charter['scenario']['criticality']}

---

## 1. Business Scenario

- **Primary Users:** {charter['scenario']['primary_users']}
- **Data Sensitivity:** {charter['scenario']['data_sensitivity']}
- **Deployment:** {charter['scenario']['deployment_target']}

### Capabilities
{''.join([f'- {c}' + chr(10) for c in charter['scenario']['capabilities']])}

## 2. High-Risk Failure Modes

| ID | Failure | Severity | Evaluator Category |
|----|---------|----------|-------------------|
{''.join([f"| {fm['id']} | {fm['failure']} | {fm['severity']} | {fm['evaluator_category']} |" + chr(10) for fm in failure_modes])}

## 3. Release Criteria

### Blocking Gates (Must Pass)
{''.join([f"- **{g['evaluator']}**: {g['threshold']} -> {g['action_on_fail']}" + chr(10) for g in release_criteria['blocking_gates']])}

### Advisory Gates (Track & Report)
{''.join([f"- **{g['evaluator']}**: {g['threshold']} -> {g['action_on_fail']}" + chr(10) for g in release_criteria['advisory_gates']])}

## 4. Evaluation Lifecycle

{''.join([f'{i+1}. {phase}' + chr(10) for i, phase in enumerate(charter['evaluation_lifecycle']['phases'])])}

## 5. Review Cadence

{charter['review_cadence']}
"""

with open("./evaluation-charter.md", "w", encoding="utf-8") as f:
    f.write(charter_md)

print("✅ Markdown charter saved to ./evaluation-charter.md")
print("\n" + "="*60)
print(charter_md[:1000] + "...")

## ✅ Lab 01 Complete

### What You Created
- **Evaluation Strategy Charter** (JSON + Markdown)
- Defined business scenario and failure modes
- Prioritized quality dimensions
- Set release criteria with blocking/advisory gates
- Established RACI ownership

### Next Steps
- **Lab 02**: Build a versioned golden dataset based on these failure modes
- **Lab 03**: Map each failure mode to specific evaluators with thresholds

### Customer Deliverable
Share `evaluation-charter.md` with stakeholders for review and approval before proceeding.